# Hafta 13: Büyük Veri Teknolojileri Uygulaması

Bu notebook'ta aşağıdaki konular ele alınmaktadır:
1. Büyük veri kavramı ve 3V/5V örnekleri
2. MapReduce modelinin Python ile simülasyonu (Word Count)
3. Apache Spark kurulumu ve SparkSession
4. Spark DataFrame işlemleri
5. Spark SQL sorguları
6. Spark MLlib ile sınıflandırma (Logistic Regression)
7. Spark MLlib ile kümeleme (K-Means)
8. Performans/ölçeklenebilirlik notları

## 1. Gerekli Kütüphaneler

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# PySpark kontrolü
try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, desc, avg
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.classification import LogisticRegression
    from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
    from pyspark.ml.clustering import KMeans
    HAS_PYSPARK = True
except Exception:
    HAS_PYSPARK = False

print('PySpark durumu:', 'Hazır' if HAS_PYSPARK else 'Eksik (pip install pyspark)')

## 2. 3V / 5V Kısa Demonstrasyon

Bu bölümde küçük bir tablo üzerinden 3V/5V kavramlarını somutlaştırıyoruz.

In [ ]:
big_data_examples = pd.DataFrame({
    'V': ['Volume', 'Velocity', 'Variety', 'Veracity', 'Value'],
    'Açıklama': [
        'Veri hacmi (TB/PB)',
        'Veri akış hızı (stream)',
        'Farklı veri tipleri (metin, görsel, log)',
        'Veri kalitesi/güvenilirlik',
        'İş değeri üretme'
    ],
    'Örnek': [
        'Günlük 2 TB işlem logu',
        'Saniyede 50k olay',
        'CSV + JSON + görsel',
        'Eksik/yanlış kayıtlar',
        'Tahmin modeli ile gelir artışı'
    ]
})

display(big_data_examples)

## 3. MapReduce Simülasyonu (Word Count)

Map ve Reduce adımlarını saf Python ile simüle ediyoruz.

In [ ]:
documents = [
    'Hello World',
    'Hello Hadoop',
    'Hello Spark Spark',
    'Big data with Spark and Hadoop'
]

# Map phase
mapped = []
for doc_id, line in enumerate(documents):
    for word in line.lower().split():
        mapped.append((word, 1))

print('Map çıktısı (ilk 15 kayıt):')
print(mapped[:15])

# Shuffle & Sort phase
shuffle = {}
for word, count in mapped:
    shuffle.setdefault(word, []).append(count)

print('\nShuffle çıktısı:')
for k in sorted(shuffle.keys()):
    print(k, shuffle[k])

# Reduce phase
reduced = {word: sum(counts) for word, counts in shuffle.items()}

print('\nReduce sonucu (word count):')
for k, v in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
    print(f'{k}: {v}')

## 4. SparkSession Başlatma

PySpark yüklüyse local modda Spark başlatılır.

In [ ]:
spark = None
if HAS_PYSPARK:
    spark = SparkSession.builder \
        .master('local[*]') \
        .appName('Hafta13-BuyukVeri') \
        .config('spark.driver.memory', '2g') \
        .getOrCreate()

    print('Spark sürümü:', spark.version)
    print('Spark başarıyla başlatıldı.')
else:
    print('PySpark yüklü değil. Spark bölümleri için: pip install pyspark')

## 5. Spark DataFrame İşlemleri

`customer_data.csv` üzerinde temel dönüşümler gösterilir.

In [ ]:
BASE = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '..', 'Veri_Setleri_Genel')
customer_path = os.path.join(BASE, 'customer_data.csv')

if spark is not None:
    sdf_customer = spark.read.csv(customer_path, header=True, inferSchema=True)

    print('Spark DataFrame şeması:')
    sdf_customer.printSchema()

    print('İlk 5 satır:')
    sdf_customer.show(5, truncate=False)

    print('Geliri 60k üzeri olanlar:')
    sdf_customer.filter(col('Annual_Income') > 60000).select('CustomerID', 'Gender', 'Annual_Income').show(10)

    print('Yaşa göre ortalama spending score:')
    sdf_customer.groupBy('Age').agg(avg('Spending_Score').alias('avg_spending')).orderBy(desc('avg_spending')).show(10)
else:
    pdf_customer = pd.read_csv(customer_path)
    display(pdf_customer.head())
    display(pdf_customer[pdf_customer['Annual_Income'] > 60000][['CustomerID', 'Gender', 'Annual_Income']].head(10))
    display(pdf_customer.groupby('Age', as_index=False)['Spending_Score'].mean().sort_values('Spending_Score', ascending=False).head(10))

## 6. Spark SQL Örneği

In [ ]:
if spark is not None:
    sdf_customer.createOrReplaceTempView('customers')

    query = '''
    SELECT Gender, COUNT(*) AS cnt, ROUND(AVG(Annual_Income), 2) AS avg_income
    FROM customers
    GROUP BY Gender
    ORDER BY cnt DESC
    '''

    spark.sql(query).show()
else:
    print('Spark yok; SQL örneği atlandı.')

## 7. Spark MLlib: Logistic Regression

`heart_disease.csv` üzerinde binary sınıflandırma uygulanır.

In [ ]:
heart_path = os.path.join(BASE, 'heart_disease.csv')

if spark is not None:
    sdf_heart = spark.read.csv(heart_path, header=True, inferSchema=True)

    feature_cols = [c for c in sdf_heart.columns if c != 'target']
    assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
    data = assembler.transform(sdf_heart).select('features', col('target').cast('double').alias('label'))

    train, test = data.randomSplit([0.8, 0.2], seed=42)

    lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=50)
    lr_model = lr.fit(train)
    pred = lr_model.transform(test)

    evaluator_auc = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
    auc = evaluator_auc.evaluate(pred)

    evaluator_acc = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
    acc = evaluator_acc.evaluate(pred)

    print(f'AUC: {auc:.4f}')
    print(f'Accuracy: {acc:.4f}')

    pred.select('label', 'prediction', 'probability').show(10, truncate=False)
else:
    print('Spark yok; MLlib sınıflandırma bölümü atlandı.')

## 8. Spark MLlib: K-Means

In [ ]:
if spark is not None:
    # customer_data üzerinde kümelenme
    num_cols = ['Age', 'Annual_Income', 'Spending_Score']
    assembler_km = VectorAssembler(inputCols=num_cols, outputCol='features')
    km_data = assembler_km.transform(sdf_customer).select('CustomerID', 'features')

    kmeans = KMeans(k=4, seed=42, featuresCol='features', predictionCol='cluster')
    km_model = kmeans.fit(km_data)
    km_pred = km_model.transform(km_data)

    print('K-Means Cluster Centers:')
    for i, center in enumerate(km_model.clusterCenters()):
        print(f'Cluster {i}: {center}')

    km_pred.groupBy('cluster').count().orderBy('cluster').show()
else:
    print('Spark yok; MLlib K-Means bölümü atlandı.')

## 9. Mini Benchmark: Pandas vs Spark (Opsiyonel)

Aynı işlemin (groupby ortalama) tek makine pandas ve Spark sürümleri kıyaslanır.
Not: Küçük veri setlerinde pandas genelde daha hızlı olabilir; büyük veride Spark ölçeklenir.

In [ ]:
# Pandas timing
t0 = time.time()
pdf = pd.read_csv(customer_path)
pandas_result = pdf.groupby('Gender', as_index=False)['Annual_Income'].mean()
pandas_time = time.time() - t0

spark_time = np.nan
if spark is not None:
    t1 = time.time()
    _ = sdf_customer.groupBy('Gender').agg(avg('Annual_Income').alias('avg_income')).collect()
    spark_time = time.time() - t1

bench = pd.DataFrame({
    'Engine': ['Pandas', 'Spark'],
    'Time_Seconds': [pandas_time, spark_time]
})
display(bench)

plt.figure(figsize=(6, 4))
sns.barplot(data=bench, x='Engine', y='Time_Seconds', palette='magma')
plt.title('Mini Benchmark: GroupBy Ortalama')
plt.tight_layout()
plt.show()

## 10. Sonuç

Bu uygulamada büyük veri ekosisteminin temel kavramları ve araçları pratikleştirildi:
- MapReduce akışı (map, shuffle, reduce)
- Spark DataFrame ve Spark SQL
- Spark MLlib ile sınıflandırma ve kümeleme

Özet öneri:
1. Küçük veri için pandas/scikit-learn ile başlayın.
2. Veri hacmi arttığında Spark DataFrame/MLlib'e geçin.
3. Performans için partitioning, caching ve shuffle azaltma stratejilerini kullanın.

In [ ]:
# Kaynak temizliği
if spark is not None:
    spark.stop()
    print('Spark session kapatıldı.')
else:
    print('Spark session açılmadı.')